# SHAPE Metric Summary by Model and Correctness

Generates the descriptive statistics table for SHAPE metrics broken down by model and correctness.

- rows: `model_id`
- column groups: Correct (C), Incorrect (I), Overall (O)
- metrics: `N_space_eff`, `N_trans_eff`, `ρ` (transition ratio = N_trans_eff / N_space_eff), Accuracy

## 1. Configuration

Set the three paths below to match your environment:

- `SHAPE_BASE`: base directory containing SHAPE-tagged trajectory JSONs
- `SHAPE_LABELER`: labeler subdirectory name
- `CORR_BASE`: directory containing per-model correctness JSONs

Output files are saved under `OUTPUT_DIR`.

In [1]:
import json
import math
import os
import re
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Paths resolved relative to the notebook directory (assumes CWD = notebook dir)
_NB_DIR    = Path(os.path.abspath(''))
SHAPE_BASE = str(_NB_DIR / "../../math-skills-llm/data/llm-label-semantic-space/thinkarm")
SHAPE_LABELER = "Qwen-Qwen3.5-27B"
CORR_BASE  = str(_NB_DIR / "../../ThinkARM/data/correct")

OUTPUT_DIR = Path("./shape_metric_summary_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TAXONOMY = [
    "H1", "H2", "H3", "H4", "H5", "H6", "H7", "H8",
    "H9", "H10", "H11", "N1", "N2", "N3", "N4",
]
H_IDX = {h: i for i, h in enumerate(TAXONOMY)}
N_H   = len(TAXONOMY)

MAIN_TABLE_METRICS = ["N_space_eff", "N_trans_eff", "Density"]
CORRECTNESS_ORDER  = ["Correct", "Incorrect", "Overall"]

# Paper-aligned model ordering (matches table row order)
MODEL_ORDER = [
    "Qwen3_32B",
    "deepseekR1",
    "QwQ32B",
    "DSQwen32B",
    "DeepSeek-R1-Distill-Qwen-7B",
    "DeepSeek-R1-Distill-Qwen-1.5B",
    "Phi4R",
    "Qwen3_32BNR",
    "gemini2.0flash",
    "Phi4",
    "Qwen25_32B",
    "gpt4o",
    "gemini2.5flash",
    "o3mini",
    "o1mini",
]

MODEL_DISPLAY = {
    "Qwen3_32B":                     "Qwen3-32B",
    "deepseekR1":                    "DeepSeek-R1",
    "QwQ32B":                        "QwQ-32B",
    "DSQwen32B":                     "DSQwen32B",
    "DeepSeek-R1-Distill-Qwen-7B":   "DeepSeek-R1-Distill-Qwen-7B",
    "DeepSeek-R1-Distill-Qwen-1.5B": "R1-Distill-Qwen-1.5B",
    "Phi4R":                         "Phi-4-Reasoning",
    "Qwen3_32BNR":                   "Qwen3-32B-NR",
    "gemini2.0flash":                "Gemini-2.0-Flash",
    "Phi4":                          "Phi-4",
    "Qwen25_32B":                    "Qwen-2.5-32B",
    "gpt4o":                         "GPT-4o",
    "gemini2.5flash":                "Gemini-2.5-Flash",
    "o3mini":                        "GPT-o3-mini",
    "o1mini":                        "GPT-o1-mini",
}

MODEL_CATEGORY = {
    "Qwen3_32B":                     "Open-source reasoning (full traces)",
    "deepseekR1":                    "Open-source reasoning (full traces)",
    "QwQ32B":                        "Open-source reasoning (full traces)",
    "DSQwen32B":                     "Open-source reasoning (full traces)",
    "DeepSeek-R1-Distill-Qwen-7B":   "Open-source reasoning (full traces)",
    "DeepSeek-R1-Distill-Qwen-1.5B": "Open-source reasoning (full traces)",
    "Phi4R":                         "Open-source reasoning (full traces)",
    "Qwen3_32BNR":                   "Instruction-tuned (no extended reasoning)",
    "gemini2.0flash":                "Instruction-tuned (no extended reasoning)",
    "Phi4":                          "Instruction-tuned (no extended reasoning)",
    "Qwen25_32B":                    "Instruction-tuned (no extended reasoning)",
    "gpt4o":                         "Instruction-tuned (no extended reasoning)",
    "gemini2.5flash":                "Proprietary reasoning (hidden traces)",
    "o3mini":                        "Proprietary reasoning (hidden traces)",
    "o1mini":                        "Proprietary reasoning (hidden traces)",
}

print("Config OK")
print("Output directory:", OUTPUT_DIR.resolve())

Config OK
Output directory: /data3/jongsong/SHAPE/analaysis_notebook/shape_metric_summary_outputs


## 2. SHAPE metric functions

- `N_space_eff = exp(H_space)`: effective number of semantic spaces
- `N_trans_eff = exp(H_trans) - 1`: effective number of semantic-space transitions
- `ρ = N_trans_eff / N_space_eff`: transition ratio (computed after loading)

Trajectories with no heuristic tags are excluded.

In [2]:
from shape_metrics import compute_shape_metrics

print("Metric functions loaded from shape_metrics.py")

Metric functions loaded from shape_metrics.py


## 3. Load trajectories and compute trace-level metrics

`df_raw` is a trajectory-level dataframe; each row is one `(model_id, problem_id)` trajectory.

Key columns: `model_id`, `problem_id`, `is_correct`, `cot_length`, `N_space_eff`, `N_trans_eff`, `Density` (= ρ)

In [3]:
def _find_available_models(shape_base: str, shape_labeler: str, corr_base: str) -> list[str]:
    shape_base = Path(shape_base)
    corr_base  = Path(corr_base)
    if not shape_base.exists():
        raise FileNotFoundError(f"SHAPE_BASE does not exist: {shape_base}")
    if not corr_base.exists():
        raise FileNotFoundError(f"CORR_BASE does not exist: {corr_base}")
    models = []
    for m in os.listdir(shape_base):
        shape_dir = shape_base / m / shape_labeler
        corr_file = corr_base / f"{m}.json"
        if shape_dir.is_dir() and corr_file.is_file():
            models.append(m)
    return sorted(models)


def _lookup_correctness(corr_data: dict, pid: str):
    candidates = [pid]
    try:
        candidates.append(str(int(pid)))
        candidates.append(int(pid))
    except (ValueError, TypeError):
        pass
    for key in candidates:
        if key in corr_data:
            return corr_data[key]
    return None


def load_shape_metric_dataframe(
    shape_base: str = SHAPE_BASE,
    shape_labeler: str = SHAPE_LABELER,
    corr_base: str = CORR_BASE,
    model_filter: list[str] | None = None,
) -> pd.DataFrame:
    """Load SHAPE-tagged trajectories and compute N_space_eff, N_trans_eff per trajectory."""
    models = _find_available_models(shape_base, shape_labeler, corr_base)
    if model_filter is not None:
        model_filter = set(model_filter)
        models = [m for m in models if m in model_filter]

    print(f"Detected {len(models)} models")
    for m in models:
        print(" -", m)

    records = []
    skipped_no_tags = skipped_no_corr = skipped_bad_json = 0

    for model in models:
        shape_dir = Path(shape_base) / model / shape_labeler
        corr_file = Path(corr_base) / f"{model}.json"
        with open(corr_file, "r") as f:
            corr_data = json.load(f)

        for fname in sorted(os.listdir(shape_dir)):
            if not fname.endswith(".json"):
                continue
            pid = fname[:-5]
            correct_val = _lookup_correctness(corr_data, pid)
            if correct_val is None:
                skipped_no_corr += 1
                continue
            try:
                with open(shape_dir / fname, "r") as f:
                    sh_units = json.load(f)
            except Exception:
                skipped_bad_json += 1
                continue

            metrics = compute_shape_metrics(sh_units)
            if metrics is None:
                skipped_no_tags += 1
                continue

            records.append({
                "model_id":   model,
                "problem_id": pid,
                "is_correct": int(bool(correct_val)),
                "cot_length": len(sh_units),
                **metrics,
            })

    df_raw = pd.DataFrame(records)
    if df_raw.empty:
        raise RuntimeError("No trajectories loaded. Check SHAPE_BASE, SHAPE_LABELER, and CORR_BASE.")

    df_raw["correctness"] = np.where(df_raw["is_correct"] == 1, "Correct", "Incorrect")
    df_raw["Density"] = np.where(
        df_raw["N_space_eff"] > 0,
        df_raw["N_trans_eff"] / df_raw["N_space_eff"],
        np.nan,
    )

    print(f"\nLoaded trajectories: {len(df_raw):,}")
    print(f"Skipped: {skipped_no_tags:,} no heuristic tags, "
          f"{skipped_no_corr:,} no correctness, {skipped_bad_json:,} bad JSON")
    print(f"Overall accuracy: {df_raw['is_correct'].mean():.3f}")
    return df_raw


df_raw = load_shape_metric_dataframe()
display(df_raw.head())

trace_level_path = OUTPUT_DIR / "shape_metrics_trace_level.csv"
df_raw.to_csv(trace_level_path, index=False)
print("Saved:", trace_level_path)

Detected 15 models
 - DSQwen32B
 - DeepSeek-R1-Distill-Qwen-1.5B
 - DeepSeek-R1-Distill-Qwen-7B
 - Phi4
 - Phi4R
 - QwQ32B
 - Qwen25_32B
 - Qwen3_32B
 - Qwen3_32BNR
 - deepseekR1
 - gemini2.0flash
 - gemini2.5flash
 - gpt4o
 - o1mini
 - o3mini

Loaded trajectories: 1,500
Skipped: 0 no heuristic tags, 0 no correctness, 0 bad JSON
Overall accuracy: 0.442


,model_id,problem_id,is_correct,cot_length,N_space_eff,N_trans_eff,correctness,Density
0,DSQwen32B,1,1,23,2.3659,1.3659,Correct,0.5773
1,DSQwen32B,10,0,99,5.8151,7.4169,Incorrect,1.2755
2,DSQwen32B,100,0,112,1.0000,0.0000,Incorrect,0.0000
3,DSQwen32B,11,1,12,1.2507,0.2507,Correct,0.2005
4,DSQwen32B,12,1,19,1.5070,0.5070,Correct,0.3364


Saved: shape_metric_summary_outputs/shape_metrics_trace_level.csv


## 4. Model order and counts

Apply `MODEL_ORDER` to produce the paper-aligned row ordering. Models absent from `MODEL_ORDER` are appended alphabetically.

In [4]:

def apply_model_order(df: pd.DataFrame, model_order: list[str] | None = None) -> pd.DataFrame:
    models_present = list(df["model_id"].drop_duplicates())
    if model_order is None:
        ordered = sorted(models_present)
    else:
        ordered = [m for m in model_order if m in models_present]
        ordered += sorted([m for m in models_present if m not in ordered])
    out = df.copy()
    out["model_id"] = pd.Categorical(out["model_id"], categories=ordered, ordered=True)
    return out.sort_values("model_id")


df_raw = apply_model_order(df_raw, MODEL_ORDER)

count_table = pd.DataFrame(index=df_raw["model_id"].cat.categories)
count_table["Correct"] = df_raw[df_raw["is_correct"] == 1].groupby("model_id", observed=False).size()
count_table["Incorrect"] = df_raw[df_raw["is_correct"] == 0].groupby("model_id", observed=False).size()
count_table["Overall"] = df_raw.groupby("model_id", observed=False).size()
count_table["Accuracy"] = df_raw.groupby("model_id", observed=False)["is_correct"].mean()
count_table = count_table.dropna(how="all").fillna(0)
count_table[["Correct", "Incorrect", "Overall"]] = count_table[["Correct", "Incorrect", "Overall"]].astype(int)

count_display = count_table.copy()
count_display.index = [MODEL_DISPLAY.get(str(m), str(m)) for m in count_display.index]

display(
    count_display.style
        .format({"Accuracy": "{:.3f}"})
        .set_caption("Number of trajectories by model and correctness")
)

count_path = OUTPUT_DIR / "shape_metric_summary_counts.csv"
count_table.to_csv(count_path)
print("Saved:", count_path)


,Correct,Incorrect,Overall,Accuracy
Qwen3-32B,63,37,100,0.630
DeepSeek-R1,63,37,100,0.630
QwQ-32B,63,37,100,0.630
DSQwen32B,57,43,100,0.570
DeepSeek-R1-Distill-Qwen-7B,49,51,100,0.490
R1-Distill-Qwen-1.5B,36,64,100,0.360
Phi-4-Reasoning,37,63,100,0.370
Qwen3-32B-NR,34,66,100,0.340
Gemini-2.0-Flash,34,66,100,0.340
Phi-4,29,71,100,0.290


Saved: shape_metric_summary_outputs/shape_metric_summary_counts.csv


## 5. Main table: model × correctness × metric

Builds the wide summary table corresponding to the paper's descriptive statistics table (`N_space_eff`, `N_trans_eff`, `ρ`, Accuracy). Default aggregation: mean.

In [5]:
AGG_FUNC     = "mean"
ROUND_DIGITS = 2


def make_metric_summary_wide(
    df: pd.DataFrame,
    metrics: list[str] = MAIN_TABLE_METRICS,
    agg_func: str = "mean",
    correctness_order: list[str] = CORRECTNESS_ORDER,
) -> pd.DataFrame:
    """Return wide table: rows=model_id, columns=(Correctness, Metric)."""
    if agg_func not in {"mean", "median"}:
        raise ValueError("agg_func must be 'mean' or 'median'")
    tables = {}
    for group_name in correctness_order:
        sub     = df if group_name == "Overall" else df[df["correctness"] == group_name]
        grouped = sub.groupby("model_id", observed=False)[metrics]
        tables[group_name] = grouped.mean() if agg_func == "mean" else grouped.median()
    wide = pd.concat(tables, axis=1)
    wide.columns.names = ["Correctness", "Metric"]
    return wide


metric_summary    = make_metric_summary_wide(df_raw, metrics=MAIN_TABLE_METRICS, agg_func=AGG_FUNC)
accuracy_by_model = df_raw.groupby("model_id", observed=False)["is_correct"].mean()

# Swap column hierarchy to (Metric, Correctness)
metric_summary = metric_summary.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0, sort_remaining=False)

category_rank = {
    "Open-source reasoning (full traces)":      0,
    "Instruction-tuned (no extended reasoning)": 1,
    "Proprietary reasoning (hidden traces)":     2,
}
order_map = {m: i for i, m in enumerate(MODEL_ORDER)}

ordered_models = (
    pd.DataFrame({"model_id": metric_summary.index.astype(str)})
    .assign(
        category      = lambda d: d["model_id"].map(MODEL_CATEGORY).fillna("Uncategorized"),
        accuracy      = lambda d: d["model_id"].map(accuracy_by_model).fillna(np.nan),
        cat_rank      = lambda d: d["category"].map(category_rank).fillna(999),
        order         = lambda d: d["model_id"].map(order_map).fillna(999),
        display_model = lambda d: d["model_id"].map(MODEL_DISPLAY).fillna(d["model_id"]),
    )
    .sort_values(["cat_rank", "order"])
)

ordered_ids    = ordered_models["model_id"].tolist()
metric_summary = metric_summary.loc[ordered_ids]

# Append single accuracy column
metric_summary[("accuracy", "Overall")] = (
    ordered_models.set_index("model_id").loc[ordered_ids, "accuracy"].values
)

metric_order  = ["N_space_eff", "N_trans_eff", "Density", "accuracy"]
ordered_cols  = [
    (m, c) for m in metric_order for c in CORRECTNESS_ORDER
    if (m, c) in metric_summary.columns
]
metric_summary = metric_summary.loc[:, ordered_cols]

row_index = pd.MultiIndex.from_arrays(
    [
        ordered_models.set_index("model_id").loc[ordered_ids, "category"],
        ordered_models.set_index("model_id").loc[ordered_ids, "display_model"],
    ],
    names=["Category", "Model"],
)
metric_summary_display       = metric_summary.copy()
metric_summary_display.index = row_index

display(
    metric_summary_display.style
        .format(f"{{:.{ROUND_DIGITS}f}}")
        .set_caption(f"SHAPE metric summary by model / correctness ({AGG_FUNC})")
)